# MB1 v0.2.1 — Supplementary Boundary-Rich Candidate Mining (Mode A only)

Preserves the 13 MB1 v0.2 USABLE seeds and mines NEW model-free evidence with coarse full-video scanning plus mandatory candidate-local hard/soft-cut verification. No semantic labels, interval GT, CLIP, VLM, optical flow, tracking, or model download.

Required Kaggle inputs:
1. Raw AIC dataset: `/kaggle/input/datasets/nadkli/dataset-aic`
2. MB1 v0.2 pack: `/kaggle/input/datasets/irthn1311/triage-eg-mb1-v02-candidates`
3. MB1 v0.2 AI-QC Pass-1: `/kaggle/input/datasets/irthn1311/mb1-v02-ai-qc-pass1-bundle`
4. RT2 AI benchmark: `/kaggle/input/datasets/irthn1311/triage-eg-rt2-ai-benchmark-bundle`

Optional/recommended input:
5. Prior RT2 Mode-A candidates: `/kaggle/input/datasets/irthn1311/triage-eg-rt2-benchmark-candidates`

Repository cloning is the only step that may require Internet. The experiment itself is offline. Output ZIP: `/kaggle/working/triage_eg_mb1_v021_candidates.zip`.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from zipfile import ZipFile

REPO_URL = os.environ.get('AIC_REPO_URL', 'https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git')
REPO_REF = os.environ.get('AIC_REPO_REF', 'TRIAGEEG')
REPO_DIR = Path(os.environ.get('AIC_REPO_DIR', '/kaggle/working/AIC2026_TeamPTK_SGU'))
if not (REPO_DIR / 'src/triage_eg').is_dir():
    if REPO_DIR.exists():
        raise RuntimeError(f'Incomplete repository directory: {REPO_DIR}')
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
sys.path.insert(0, str(REPO_DIR / 'src'))
print({'resolved_repo': str(REPO_DIR), 'ref': REPO_REF, 'commit': commit})

In [ ]:
DATASET_INPUT = Path(os.environ.get('AIC_DATA_ROOT', '/kaggle/input/datasets/nadkli/dataset-aic'))
OLD_INPUT = Path(os.environ.get('AIC_MB1_V02_ROOT', '/kaggle/input/datasets/irthn1311/triage-eg-mb1-v02-candidates'))
QC_INPUT = Path(os.environ.get('AIC_MB1_V02_QC_ROOT', '/kaggle/input/datasets/irthn1311/mb1-v02-ai-qc-pass1-bundle'))
RT2_INPUT = Path(os.environ.get('AIC_RT2_ROOT', '/kaggle/input/datasets/irthn1311/triage-eg-rt2-ai-benchmark-bundle'))
RT2_MODE_A_INPUT = Path(os.environ.get('AIC_RT2_MODE_A_ROOT', '/kaggle/input/datasets/irthn1311/triage-eg-rt2-benchmark-candidates'))
OUTPUT_ROOT = Path('/kaggle/working/triage_eg_mb1_v021_candidates')
ZIP_PATH = Path('/kaggle/working/triage_eg_mb1_v021_candidates.zip')
RESOLVED_INPUT_ROOT = Path('/kaggle/working/triage_eg_mb1_v021_resolved_inputs')
print({'dataset': str(DATASET_INPUT), 'old_mb1': str(OLD_INPUT), 'qc': str(QC_INPUT), 'rt2': str(RT2_INPUT), 'rt2_mode_a': str(RT2_MODE_A_INPUT), 'output': str(OUTPUT_ROOT)})

In [ ]:
SEARCH_ROOT = Path('/kaggle/input')
MAX_DEPTH = 6
MAX_DIRECTORIES = 5000
MAX_ZIPS = 100

def bounded_directories(root: Path):
    frontier, visited = [(Path(root), 0)], 0
    while frontier:
        current, depth = frontier.pop(0)
        if not current.is_dir():
            continue
        visited += 1
        if visited > MAX_DIRECTORIES:
            raise RuntimeError('Kaggle discovery exceeded directory bound')
        yield current
        if depth < MAX_DEPTH:
            frontier.extend((item, depth + 1) for item in sorted(current.iterdir()) if item.is_dir() and not item.is_symlink())

def artifact_valid(data: bytes, filename: str, role: str):
    try:
        first = json.loads(next(line for line in data.decode('utf-8').splitlines() if line.strip())) if filename.endswith('.jsonl') else json.loads(data)
    except Exception:
        return False
    if role == 'old_manifest': return str(first.get('candidate_id', '')).startswith('mb1v02_')
    if role == 'old_diagnostics': return str(first.get('candidate_id', '')).startswith('mb1v02_')
    if role == 'old_selection': return first.get('experiment') == 'MB1_V02'
    if role == 'qc': return str(first.get('candidate_id', '')).startswith('mb1v02_') and 'qc_status' in first
    if role == 'qc_summary': return first.get('status') == 'AI_QC_PASS1_COMPLETE'
    if role == 'rt2_benchmark': return str(first.get('query_id', '')).startswith('rt2_')
    if role == 'rt2_selection': return first.get('reference_experiment') == 'RT2'
    return True

def resolve_artifact(root: Path, filename: str, role: str, *, required=True):
    base = root if root.exists() else SEARCH_ROOT
    directories = list(bounded_directories(base))
    matches = []
    direct = root if root.is_file() and root.name == filename else root / filename
    if direct.is_file() and artifact_valid(direct.read_bytes(), filename, role): matches.append(direct.resolve())
    for directory in directories:
        candidate = directory / filename
        if candidate.is_file() and artifact_valid(candidate.read_bytes(), filename, role): matches.append(candidate.resolve())
    matches = sorted(set(matches))
    if len(matches) == 1: return matches[0]
    if len(matches) > 1: raise RuntimeError(f'Ambiguous {role}: {matches}')
    zip_hits = []
    zip_paths = sorted({path.resolve() for directory in directories for path in directory.glob('*.zip')})[:MAX_ZIPS]
    for archive_path in zip_paths:
        with ZipFile(archive_path) as archive:
            for member in archive.namelist():
                if Path(member).name == filename:
                    data = archive.read(member)
                    if artifact_valid(data, filename, role): zip_hits.append((archive_path, member, data))
    if len(zip_hits) == 1:
        RESOLVED_INPUT_ROOT.mkdir(parents=True, exist_ok=True)
        target = RESOLVED_INPUT_ROOT / filename
        target.write_bytes(zip_hits[0][2])
        return target.resolve()
    if not zip_hits and not required: return None
    raise RuntimeError(f'Expected exactly one {role}/{filename}; direct={matches}, zip_hits={[(str(a), m) for a, m, _ in zip_hits]}')

def resolve_dataset(root: Path):
    candidates = [directory for directory in bounded_directories(root if root.exists() else SEARCH_ROOT) if any(path.is_dir() for path in directory.glob('Videos_*'))]
    candidates = sorted(set(path.resolve() for path in candidates))
    if len(candidates) != 1: raise RuntimeError(f'Expected exactly one raw dataset root; found {candidates}')
    return candidates[0]

In [ ]:
DATASET_ROOT = resolve_dataset(DATASET_INPUT)
OLD_MANIFEST = resolve_artifact(OLD_INPUT, 'mb1_v02_candidate_manifest.jsonl', 'old_manifest')
OLD_DIAGNOSTICS = resolve_artifact(OLD_INPUT, 'mb1_v02_candidate_diagnostics.jsonl', 'old_diagnostics')
OLD_SELECTION = resolve_artifact(OLD_INPUT, 'candidate_selection.json', 'old_selection')
AI_QC = resolve_artifact(QC_INPUT, 'mb1_v02_ai_qc_pass1.jsonl', 'qc')
AI_QC_SUMMARY = resolve_artifact(QC_INPUT, 'mb1_v02_ai_qc_summary.json', 'qc_summary')
RT2_BENCHMARK = resolve_artifact(RT2_INPUT, 'rt2_ai_benchmark.jsonl', 'rt2_benchmark')
PRIOR_RT2_SELECTION = resolve_artifact(RT2_MODE_A_INPUT, 'candidate_selection.json', 'rt2_selection', required=False)
RESOLVED = {'dataset_root': DATASET_ROOT, 'old_manifest': OLD_MANIFEST, 'old_diagnostics': OLD_DIAGNOSTICS, 'old_selection': OLD_SELECTION, 'ai_qc': AI_QC, 'ai_qc_summary': AI_QC_SUMMARY, 'rt2_benchmark': RT2_BENCHMARK, 'prior_rt2_selection': PRIOR_RT2_SELECTION}
print(json.dumps({key: str(value) if value else None for key, value in RESOLVED.items()}, indent=2))

In [ ]:
from triage_eg.experiments.mb1_v021 import MB1V021Config, preflight_mb1_v021

if OUTPUT_ROOT.exists():
    if OUTPUT_ROOT.parent != Path('/kaggle/working'): raise RuntimeError(f'Refusing cleanup outside /kaggle/working: {OUTPUT_ROOT}')
    shutil.rmtree(OUTPUT_ROOT)
ZIP_PATH.unlink(missing_ok=True)
CONFIG = MB1V021Config(dataset_root=DATASET_ROOT, old_candidate_manifest_path=OLD_MANIFEST, old_candidate_diagnostics_path=OLD_DIAGNOSTICS, old_candidate_selection_path=OLD_SELECTION, ai_qc_path=AI_QC, ai_qc_summary_path=AI_QC_SUMMARY, rt2_benchmark_path=RT2_BENCHMARK, prior_rt2_selection_path=PRIOR_RT2_SELECTION, output_root=OUTPUT_ROOT, build_git_commit=commit)
PREFLIGHT = preflight_mb1_v021(CONFIG)
print(json.dumps(PREFLIGHT, indent=2))
assert PREFLIGHT['frozen_seed_count'] == 13
assert PREFLIGHT['model_inference_required'] is False
assert PREFLIGHT['semantic_interval_gt_required'] is False

In [ ]:
from triage_eg.experiments.mb1_v021 import prepare_mb1_v021_candidates

RESULT = prepare_mb1_v021_candidates(CONFIG)
AUDIT = RESULT['audit']
print('OLD-QC CUT-GUARD REGRESSION AUDIT')
print(json.dumps({key: AUDIT[key] for key in ('old_ai_qc_hard_cut_count', 'old_hard_cut_vetoed_by_hard_cut', 'old_hard_cut_vetoed_by_soft_cut', 'old_hard_cut_vetoed_by_either', 'OLD_HARD_CUT_RECALL', 'old_usable_count', 'old_usable_falsely_vetoed', 'OLD_USABLE_FALSE_VETO_RATE')}, indent=2))
print('Thresholds were fixed before this post-hoc audit and were not auto-tuned.')

In [ ]:
SELECTION = RESULT['selection']
RUN = RESULT['run_manifest']
print(json.dumps({'source_videos': len(SELECTION['source_videos_considered']), 'existing_seeds': SELECTION['existing_seed_count'], 'raw_proposals': SELECTION['raw_proposals_generated'], 'rejected': SELECTION['rejected'], 'retained_NEW': SELECTION['retained_NEW_candidate_count'], 'combined_potential_pool': SELECTION['combined_potential_pool_count'], 'per_video': SELECTION['per_video_retained_counts'], 'rescued_prior_regions': SELECTION['rescued_prior_region_count'], 'runtime_seconds': RUN['performance']['overall_runtime_ms'] / 1000}, indent=2))
rows = [json.loads(line) for line in (OUTPUT_ROOT / 'mb1_v021_candidate_manifest.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
assert all(row['continuity_status'] == 'PASS_LOCAL_HARD_AND_SOFT_CUT_GUARD' for row in rows)
assert all(row['overview_displayed_frames'] == sorted(set(row['overview_displayed_frames'])) and row['dense_displayed_frames'] == sorted(set(row['dense_displayed_frames'])) for row in rows)
assert all((OUTPUT_ROOT / row['overview_sheet_path']).is_file() and (OUTPUT_ROOT / row['dense_sheet_path']).is_file() for row in rows)
print('manifest/sheet/raw-frame mapping: EXACT')

In [ ]:
from IPython.display import Image, display

for row in rows[:3]:
    print(row['candidate_id'], row['video_id'], row['window_adjustment_reason'], row['candidate_origin'])
    display(Image(filename=str(OUTPUT_ROOT / row['overview_sheet_path'])))
    display(Image(filename=str(OUTPUT_ROOT / row['dense_sheet_path'])))
montages = sorted((OUTPUT_ROOT / 'montages').glob('overview_montage_*.jpg'))
if montages: display(Image(filename=str(montages[0])))

In [ ]:
from triage_eg.experiments.mb1_v021 import create_mb1_v021_bundle

archive = create_mb1_v021_bundle(OUTPUT_ROOT, ZIP_PATH)
with ZipFile(archive) as stream:
    members = stream.namelist()
assert not any(name.endswith(('.mp4', '.npy', '.npz', '.pt', '.pth', '.bin')) for name in members)
print('DOWNLOAD ZIP:', archive)
print('size_bytes:', archive.stat().st_size, 'members:', len(members))
print('MB1_V021_REAL_STATUS = COMPLETE')
print('MB1_V021_AI_QC_STATUS = WAITING_FOR_AI')
print('M3_IMPLEMENTATION_STATUS = NOT_STARTED')